In [8]:
%load_ext autoreload
%autoreload 2
from openai import OpenAI
from tqdm.auto import tqdm
import tiktoken
import glob
import pandas as pd
import json

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
DATA_DIR = "./golden-dataset/data/"
files = glob.glob(f"{DATA_DIR}*_dataset.csv")

df_list = [pd.read_csv(f) for f in files if "empty" not in f and "refused" not in f]
merged_df = pd.concat(df_list, axis=0, ignore_index=True)
test_df = merged_df[merged_df["split"] == "test"]

In [9]:
with open('./golden-dataset/personas_desc.json', 'r') as f:
        personas = json.load(f)

In [30]:
prompts = []
for persona_name, persona_desc in personas.items():

    persona_instruction = f"You are exactly this character: {persona_name}. {persona_desc}"
    prompts.extend([persona_instruction + q
                for q in test_df[test_df['persona'] == persona_name]["prompt"]
            ])

In [31]:
len(prompts)

2970

In [32]:
enc = tiktoken.encoding_for_model("gpt-5")

In [33]:
tokens = enc.encode_batch(prompts)

In [36]:
input_tokens = sum([len(t) for t in tokens])

In [34]:
output_tokens = 256 * len(prompts)

In [37]:
# gpt-5.6-terra:
1.25*input_tokens/1e6 + 1.5625*output_tokens/1e6

1.8997312499999999

In [38]:
# gpt-5.6-sol:
2.5*input_tokens/1e6 + 3.125*output_tokens/1e6

3.7994624999999997

In [39]:
# gpt-5.6-luna:
0.5*input_tokens/1e6 + 0.625*output_tokens/1e6

0.7598925000000001